# Inference on FPGA of HLS4ML-projects with Vitis Unified

This notebook is optimized to test performance across multiple synthesized models of the same dataset. Meaning the dataset is common, but architecture may be wildly different. It may be run in vanilla python-environment. 

It's based on [fastmachinelearning/hls4ml-tutorial/part7b_deployment.ipynb](https://github.com/fastmachinelearning/hls4ml-tutorial/blob/main/part7b_deployment.ipynb), [Tanawin1701d/vitisUnifiedTutorial/part8b_testOnHw.ipynbxt](https://github.com/Tanawin1701d/vitisUnifiedTutorial/blob/main/part8b_testOnHw.ipynb) and earlier experimentation. For syntax, see driverclass in directory and [pynq-package documentation](https://pynq.readthedocs.io/en/v2.5.1/pynq_package/pynq.overlay.html).


## Load project

Vitis Unified copies the final files needed for inference to the subfolder in HLS4ML-projectdirectory `export/`. 
Copy the bitfile (.bit) and descriptor (.hwh) into its own directory under `DUT/` (Device Under Test) with explanatory name on the directory.


In [1]:
search_dir = 'DUT/'
bitfile_pattern = '*.bit'
hwh_pattern = '*.hwh'

x_test_path = 'processed_data/x_test.npy'
y_test_path = 'processed_data/y_test.npy'

prediction_path = 'predictions/'

In [2]:
from pathlib import Path
duts = []

print(f"Searching for models to test in {search_dir}")
for bitfile_path in Path(search_dir).rglob(bitfile_pattern):
    #print(path, parent_dir)
    parent_dir = bitfile_path.parent
    if parent_dir.rglob(hwh_pattern): # make sure hwh is there as well
        # https://docs.python.org/3/library/pathlib.html#pathlib.PurePath
        project_name = f"{bitfile_path.parts[-2]}" 
        print(f"\n\n%%%%%%% Model to test: {project_name} ({bitfile_path}) %%%%%%%%%%%\n")
        duts.append({
            'project_name' : project_name,
            'bitfile_path' : bitfile_path
        })

Searching for models to test in DUT/


%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7084_ebops=864_VU_DA_bitfile (DUT/Training_AdaptiveHP_acc=0.7084_ebops=864_VU_DA_bitfile/system.bit) %%%%%%%%%%%



%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_latency_bitfile (DUT/Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_latency_bitfile/system.bit) %%%%%%%%%%%



%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7449_ebops=3640_VU_DA_bitfile (DUT/Training_AdaptiveHP_acc=0.7449_ebops=3640_VU_DA_bitfile/system.bit) %%%%%%%%%%%



%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7084_ebops=864_VU_latency_bitfile (DUT/Training_AdaptiveHP_acc=0.7084_ebops=864_VU_latency_bitfile/system.bit) %%%%%%%%%%%



%%%%%%% Model to test: Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_DA_bitfile (DUT/Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_DA_bitfile/system.bit) %%%%%%%%%%%



In [3]:
duts

[{'project_name': 'Training_AdaptiveHP_acc=0.7084_ebops=864_VU_DA_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7084_ebops=864_VU_DA_bitfile/system.bit')},
 {'project_name': 'Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_latency_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_latency_bitfile/system.bit')},
 {'project_name': 'Training_AdaptiveHP_acc=0.7449_ebops=3640_VU_DA_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7449_ebops=3640_VU_DA_bitfile/system.bit')},
 {'project_name': 'Training_AdaptiveHP_acc=0.7084_ebops=864_VU_latency_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7084_ebops=864_VU_latency_bitfile/system.bit')},
 {'project_name': 'Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_DA_bitfile',
  'bitfile_path': PosixPath('DUT/Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_DA_bitfile/system.bit')}]

Load datasets for testing performance in inference

In [ ]:
import numpy as np
# Load input from .npy file
x_test = np.load(x_test_path).astype('float32')
y_test = np.load(y_test_path).astype('float32')

In [ ]:
# import the library/driver which is common for every VItis Unified synthesis
from axi_master_driver import NeuralNetworkOverlay

def test_dut(bitfile_path, project_name, prediction_path):
    # create the overlay object
    overlay = NeuralNetworkOverlay(bitfile_name=bitfile_path, x_shape=x_test.shape, y_shape=y_test.shape, dtype=x_test.dtype)
    
    # Do the prediction/run inference
    result = overlay.predict(x_test, debug=False, profile=True, encode=np.float32, decode=np.float32)
    y_dut = result[0]

    # Save results
    np.save(f"{prediction_path}/y_dut_{project_name}.npy",y_dut)

    # TODO: calculate performance

    return result # contains the output buffer, execution time and inference/s

In [6]:
def cal_accuracy(y_dut):
    y_pred = np.argmax(y_dut, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return np.sum(y_pred == y_true) / len(y_true)

# test the function 
#y_dut = np.load('../testmodel_2_VitisUnifiedKV260/y_hardware.npy')
#acc = cal_accuracy(y_dut)
#print(f"accuracy of hardware inference: {acc}")

Run the actual inference

In [7]:
import os
prediction_path = 'predictions'
os.makedirs(prediction_path,exist_ok=True)

for dut in duts:
    print(f"\n\nLoading and testing {dut['project_name']}")
    print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%\n")
    
    # do inference
    result = test_dut(str(dut['bitfile_path']), dut['project_name'], prediction_path)
    y_dut = result[0] # retrieving the 
    #print(y_dut[:10])

    # calculate acc
    acc = cal_accuracy(y_dut)
    print(f"\naccuracy of hardware inference: {acc}")



Loading and testing Training_AdaptiveHP_acc=0.7084_ebops=864_VU_DA_bitfile
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%



input gmem_in0_ptr_linput will be set to addr: 0x3bd00000 with elements: 24782040
output gmem_out0_ptr_layer5_out will be set to addr: 0x3ae00000 with elements: 1239102
amount of queries will be set to: 413034 at address: 0x28
prepare your interrupt
global interrupt enable register
enable gie successful
ap_done interrupt enable register
enable ap_done interrupt successful
ap_done register clear
clear ap_done interrupt successful
----------------------
starting the accelerator
Accelerator execution time:                       1.157 ms
accelerator has finished
Processed elements: 413034, Execution time: 0.0012 seconds, Performance rate: 357002155.16 inferences/second

accuracy of hardware inference: 0.4439997675736138


Loading and testing Training_AdaptiveHP_acc=0.7573_ebops=11698_VU_latency_bitfile
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

input gmem_in0_ptr_linput will be set to addr: 0x3bd00000 with elements: 24782040
output gmem_out0_ptr_layer5_out will be set to addr: 0x41c00